# Lab 2 — Deploy a prompt agent

**Required · 50 minutes · Level 100**

## What will you do?

In Lab 1 the instructions lived in your Python file. In this lab you move them into Foundry, where they become an **agent**: a named object that carries its own model, its own instructions and its own version history.

Nothing about the model changes. What changes is who owns the behaviour.

| | Model deployment, Lab 1 | Prompt agent, this lab |
|---|---|---|
| Where the instructions live | In your Python file | In Foundry |
| Who can change them | Whoever can edit and redeploy the code | Anyone with portal access, with no redeployment |
| History of changes | Your git log, if you remembered to commit | Built in. Every save creates a new version. |
| How a request selects it | `model="my-deployment"` | An `agent_reference` holding a name and a version |
| What a reviewer reads | Source code | The agent page in the portal |

"Deploy an agent" here means creating a named, versioned prompt agent in Foundry. It does not mean deploying a container or a web app.

```text
PromptAgentDefinition  ->  create_version  ->  agent name + version  ->  agent_reference  ->  answer
```

In this lab you will:

1. Write instructions for a small workshop guide and save them as version 1.
2. Point a request at that exact version.
3. See what a conversation remembers between messages.
4. Save a second version and prove the first one did not change.

## New words

- **Agent** — a model plus saved instructions, stored under a name you choose. Optionally it also carries tools; today it has none.
- **Agent version** — one frozen definition. Saving new instructions never edits an existing version. It creates the next one.
- **Agent reference** — the small object a request uses to say *which* agent and *which* version should answer.
- **Conversation** — a server-side thread that remembers earlier messages, so a follow-up question makes sense without you resending the history.

## Before you start

- Python 3.11 or later, with a notebook kernel selected.
- `az login` completed.
- A Foundry project endpoint and a model deployment, from Lab 1.
- Permission to create agents in that project.

**How the To-Do sections work.** Replace each `...` blank, then run the cell with **Shift+Enter**. A blank you leave open stops the cell with a message naming it, before anything reaches Azure. Try the task, then the hint, then the solution.

Each run of the definition cell creates another agent version, so run it once and re-read the output rather than re-running to see it again.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0"

## 0. Connect to your project

The next cell reads your two nonsecret settings, signs in with your Azure CLI login, and reserves a unique name for the agent you are about to create. The random suffix keeps your agent separate from everyone else's in the shared project, so nobody overwrites your work and you never clean up somebody else's.

`check_todos` is a helper, not part of the exercise. It stops a cell while a `...` blank is still open.

**Run the cell. You should see** `Reserved name: day1-guide-` followed by eight random characters. Nothing exists in Foundry yet — this is only a name.

In [ ]:
import os
import sys
from uuid import uuid4

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential

# Your two nonsecret settings. Paste them here or set them as environment variables.
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")

if sys.version_info < (3, 11):
    raise RuntimeError(
        "These notebooks need Python 3.11 or later. This kernel is "
        f"{sys.version_info.major}.{sys.version_info.minor}. Select a newer kernel."
    )

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    raise ValueError(
        "Set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME, "
        "or paste your own values into the two lines above."
    )


def check_todos(**answers: object) -> None:
    """Helper. Stops the cell while a `...` blank is still open."""
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


AGENT_NAME = f"day1-guide-{uuid4().hex[:8]}"

credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client(timeout=60, max_retries=0)
print(f"Reserved name: {AGENT_NAME}")

## 1. Write the instructions and save version 1

Two objects do the work here, and it is worth keeping them straight:

- `PromptAgentDefinition(model=..., instructions=...)` is a **description**. Building it changes nothing in Azure. It is a Python object sitting in memory.
- `project.agents.create_version(agent_name=..., definition=...)` is the **call that saves it**. This is the moment your agent exists in Foundry.

Well-written agent instructions usually stack in this order, and the order matters because later lines qualify earlier ones:

```text
1. Role        who the agent is and who it serves
2. Boundary    what it may use, and what to do when it cannot answer
3. Knowledge   the facts it is allowed to state
4. Style       the shape of the answer
```

Today the knowledge is a few lines of a Foundry reference pasted straight into the instructions. That is honest for a handful of facts and completely impractical beyond that: every request pays for every fact in tokens, and updating a fact means creating a new version. Lab 3 replaces this with retrieval.

### To-Do 1 — Complete the agent definition

**Goal:** one saved agent version in Foundry that you can open in the portal.

**Steps**

1. Set `AGENT_MODEL` to the deployment this agent should run on.
2. Write `AGENT_ROLE` as one sentence giving the agent a job and an audience. This is row 1 of the stack above; the cell supplies rows 2 to 4.
3. Run the cell **once**. Each run creates another version.
4. Open the agent in the portal and find its model, its instructions and its version number.

**Predict:** someone asks this agent which Azure regions Foundry is available in, which the reference does not mention. What should it do? You will test that later in the lab.

**Run the cell. You should see** `Saved in Foundry: day1-guide-... version 1`.

<details><summary>Hint</summary>

`AGENT_MODEL` reuses a variable you already have, without quotation marks. For `AGENT_ROLE`, name both the job and who it is for — "help developers who are new to ..." is a better opening than "you are helpful".

</details>

<details><summary>Show solution code</summary>

```python
AGENT_MODEL = MODEL_DEPLOYMENT
AGENT_ROLE = (
    "You are an onboarding assistant for Microsoft Foundry. "
    "Help developers who are new to the platform get their first agent working."
)
```

</details>

In [ ]:
SERVICE_NOTES = """Microsoft Foundry quick reference (workshop extract):
- Saving an agent with create_version returns a version number. The first save is version 1.
- A saved version is immutable. Editing the instructions and saving again produces the
  next version, and every earlier version stays callable.
- A request selects a saved agent with an agent_reference holding the agent's name and
  its version number as text.
- Microsoft Agent Framework runs on your own machine and calls agents stored in Foundry.
"""

AGENT_MODEL = ...  # TODO 1: choose the deployment this agent runs on.
AGENT_ROLE = ...  # TODO 1: write the agent's job and audience in one sentence.
check_todos(AGENT_MODEL=AGENT_MODEL, AGENT_ROLE=AGENT_ROLE)

BOUNDARY = (
    "Use only the reference below. If it does not cover the question, say you do not know. "
    "Never invent an API name, a version number or a region."
)
STYLE_V1 = "Answer in one short paragraph."

BASE_INSTRUCTIONS = f"{AGENT_ROLE}\n{BOUNDARY}\n{SERVICE_NOTES}\n"

agent_v1 = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=AGENT_MODEL,
        instructions=BASE_INSTRUCTIONS + STYLE_V1,
    ),
)
print(f"Saved in Foundry: {agent_v1.name} version {agent_v1.version}")

## 2. Point a request at one exact version

Calling an agent uses the same `client.responses.create(...)` you already know, with one difference: you no longer pass `model`. The agent already knows its own model. Instead you pass an **agent reference** that names which saved definition should answer:

```python
extra_body={"agent_reference": {"type": "agent_reference", "name": ..., "version": ...}}
```

Two details trip people up.

- `create_version` hands you back an object with `.name` and `.version`. Read them from that object rather than typing them, so your code keeps working when the version number moves on.
- The service expects `version` as **text**, not as a number. Wrap it in `str(...)`.

Naming the version explicitly is a deliberate choice. It means the answer you get today is the answer you get next month, because that definition is frozen. Omit the version and you inherit whatever someone saved most recently.

The other new parameter is `conversation`. A conversation is a server-side thread. Send a follow-up with the same conversation id and the agent still has the earlier messages; use a new id and it starts from nothing.

### To-Do 2 — Build the agent reference

**Goal:** a helper that can ask *any* version of *any* agent, not just this one.

**Steps**

1. Inside `ask_agent`, set `"name"` from the function's `agent` parameter.
2. Set `"version"` from the same parameter, converted to text.
3. Run the cell. Check that the printed name and version match what you saw in the portal.

**Predict:** why should the function read from its `agent` parameter instead of referring to `agent_v1` directly? You will need the answer in section 3.

**Run the cell. You should see** the agent name, its version, `completed`, and an answer giving the 100-per-minute limit.

<details><summary>Hint</summary>

The `agent` parameter *is* the object `create_version` returned. It has `.name` and `.version`. One of the two needs `str(...)` around it.

</details>

<details><summary>Show solution code</summary>

```python
reference = {
    "type": "agent_reference",
    "name": agent.name,
    "version": str(agent.version),
}
```

</details>

In [ ]:
def ask_agent(agent, question, conversation_id):
    """Ask one specific agent version a question inside one conversation."""
    reference = {
        "type": "agent_reference",
        "name": ...,  # TODO 2: read the name from the agent parameter.
        "version": ...,  # TODO 2: read its version, as text.
    }
    check_todos(name=reference["name"], version=reference["version"])

    response = client.responses.create(
        conversation=conversation_id,
        input=question,
        extra_body={"agent_reference": reference},
    )
    print(f"[{agent.name} v{agent.version}] {response.status}")
    print(response.output_text)
    return response


QUESTION = "Which version number does an agent get the first time I save it?"

conversation_v1 = client.conversations.create()
answer_v1 = ask_agent(agent_v1, QUESTION, conversation_v1.id)

### What the conversation remembers

The next cell asks two follow-ups in that same conversation. Read them carefully — neither one makes sense on its own:

- `"And what happens to it when I save again?"` has no subject. "It" is version 1, and the agent has to resolve that from the thread.
- `"Which Azure regions is Foundry available in?"` is the test you predicted in To-Do 1. The reference deliberately says nothing about regions.

**Run the cell. You should see** a sensible answer to the first follow-up, and an admission of not knowing for the second.

The second question is the harder test, and deliberately so. A fictional API is easy to be honest about because the model has nothing to fall back on. Foundry is a real product the model saw during training, so it holds a plausible-sounding answer already — and region lists change, so a remembered one is likely to be wrong. Your boundary rule is the only thing standing between the reader and a confident, outdated answer. If the agent lists regions anyway, note where the rule broke down: that is precisely the failure mode of instruction-only guardrails.

In [ ]:
follow_up = ask_agent(agent_v1, "And what happens to it when I save again?", conversation_v1.id)
print()
gap = ask_agent(agent_v1, "Which Azure regions is Foundry available in?", conversation_v1.id)

## 3. Save a second version

Here is the part people get wrong. Editing a Python string does **not** change a saved agent. Foundry has already stored version 1 exactly as you sent it. To change behaviour you build a new definition and call `create_version` again, which returns version 2 and leaves version 1 exactly where it was.

That is the whole point of versioning:

```text
version 1  ->  frozen, still callable, still answers in the old style
version 2  ->  your new instructions
```

You can roll back by referencing version 1 again. You can compare them side by side. You can leave production on version 1 while you test version 2. None of that works if saving overwrote the previous definition.

To make the comparison honest, change exactly one thing. Keep the model, the role, the boundary and the facts identical, and change only the style line. Then use a **fresh conversation** for each version, so no earlier message influences the answer.

### To-Do 3 — Change the style and compare versions

**Goal:** two versions of one agent, answering the same question in visibly different formats.

**Steps**

1. Write `STYLE_V2` to request a different answer format from version 1's single paragraph.
2. Run the cell. It saves version 2, then asks both versions the same question in separate conversations.
3. Compare the two answers, then check that version 1 still behaves as it always did.

**Run the cell. You should see** `Saved version 2`, then version 1's paragraph, then version 2's new format.

<details><summary>Hint</summary>

Change the shape of the answer, not the job or the facts. Bullet points contrast clearly with a paragraph, which makes the difference easy to see at a glance.

</details>

<details><summary>Show solution code</summary>

```python
STYLE_V2 = "Answer with at most three short bullet points, one fact per bullet."
```

</details>

In [ ]:
STYLE_V2 = ...  # TODO 3: request a different answer format.
check_todos(STYLE_V2=STYLE_V2)

agent_v2 = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=AGENT_MODEL,
        instructions=BASE_INSTRUCTIONS + STYLE_V2,
    ),
)
print(f"Saved version {agent_v2.version}. Version {agent_v1.version} is unchanged.\n")

print("VERSION 1, asked again in a fresh conversation")
recheck_v1 = ask_agent(agent_v1, QUESTION, client.conversations.create().id)

print("\nVERSION 2")
answer_v2 = ask_agent(agent_v2, QUESTION, client.conversations.create().id)

## Deterministic success check

Model wording varies, so this check looks at structure rather than words. It confirms that both versions exist under one agent name, that every request completed, and that your two versions are genuinely different definitions.

In [ ]:
assert agent_v1.name == agent_v2.name == AGENT_NAME, "Both versions should sit under one agent name."
assert str(agent_v1.version) != str(agent_v2.version), "create_version should have produced a new version."
for label, response in [
    ("version 1", answer_v1),
    ("follow-up", follow_up),
    ("uncovered", gap),
    ("version 1 recheck", recheck_v1),
    ("version 2", answer_v2),
]:
    assert response.status == "completed", f"The {label} request did not complete."
    assert response.output_text.strip(), f"The {label} response came back empty."
print(
    f"PASS - agent {AGENT_NAME} answered from version {agent_v1.version} "
    f"and version {agent_v2.version}, and both versions still exist."
)

## What you learned

- An **agent** moves the instructions out of your code and into Foundry, where they get a name, a version and a portal page.
- `create_version` **saves**; `PromptAgentDefinition` only describes. Editing a local string changes nothing on the server.
- Versions are frozen. Referencing a version by name and number is what makes an agent's behaviour reproducible.
- A **conversation** carries context, which is why `"And what happens to it when I save again?"` resolved to version 1 at all.
- Facts pasted into instructions are paid for on every request and can only be updated by creating a new version.

**Reflection.** One sentence each.

1. Your agent gave a wrong answer in production yesterday. What do you need in order to reproduce it exactly?
2. One line of the reference becomes out of date. What has to happen, and who has to do it?
3. Did the region question get an honest answer? What does that tell you about relying on instructions for safety?

<details><summary>Compare your answers</summary>

1. The agent name *and* the version number that served the request. Without the version you can only guess, since the definition may have moved on since.
2. Someone edits the reference text and saves a new version. That is a content change, not a code change — but it is also a full version bump for one line, which is exactly why Lab 3 moves facts out of instructions.
3. Instructions steer a model; they do not constrain it. A boundary rule reduces invented answers but never guarantees their absence, and it is weakest exactly where the model already believes it knows the answer. Treat it as guidance, not as a security control.

</details>

**If something fails:** confirm the project supports prompt agents, that the model supports the Responses API, and that your login can create agents. Restart the kernel after changing packages. A `404` on the agent reference usually means the version was passed as a number rather than as text.

**Reset:** the cleanup cell closes local clients only. It does not delete anything in Azure. To remove the agent afterwards, use `project.agents.delete_version(agent_name=..., agent_version=...)` or delete it from the portal.

**Expected artifact:** one agent with two versions, and a passing success check.

**Next:** Lab 3 removes the pasted facts and gives the agent a searchable set of documents instead.

In [ ]:
client.close()
project.close()
credential.close()
print("Closed the local clients. Your agent versions remain in Foundry.")